In [1]:
"""
3. Magnitude Pruning — ResNet-50 / CIFAR-10
============================================
Per-layer (local) magnitude pruning using L1 and L2 norms.
Unlike global unstructured pruning, each layer is pruned
independently to the same target sparsity.

Compares L1 vs L2 magnitude criteria at each sparsity level.

Output: 3_magnitude_Pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "3_v2_magnitude_Pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS = [0.30]  # , 0.50, 0.70, 0.80, 0.90
MAX_ACC_DROP    = 0.02
INFERENCE_RUNS  = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Pruning ───────────────────────────────────────────────────────────────────
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def apply_magnitude_pruning_l1(model, sparsity):
    """Per-layer L1 unstructured magnitude pruning (local, not global)."""
    pruned = copy.deepcopy(model)
    for module, param in prunable_layers(pruned):
        prune.l1_unstructured(module, name=param, amount=sparsity)
        prune.remove(module, param)
    return pruned

def apply_magnitude_pruning_l2(model, sparsity):
    """Per-layer L2 magnitude pruning using random_unstructured as scaffold
    with L2-norm-based manual mask."""
    pruned = copy.deepcopy(model)
    for _, module in pruned.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.data
            flat = w.abs().pow(2).view(-1)  # L2^2 scores (monotone w/ L2)
            threshold = torch.kthvalue(flat, max(1, int(flat.numel() * sparsity))).values
            mask = (w.abs().pow(2) > threshold).float()
            module.weight.data = w * mask
    return pruned

def apply_magnitude_pruning_global(model, sparsity):
    """Global L1 magnitude pruning — all layers ranked together (reference)."""
    pruned = copy.deepcopy(model)
    layers = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
    for module, param in layers:
        prune.remove(module, param)
    return pruned

def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_gpu_ms(model):
    if DEVICE.type != "cuda": return -1.0
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(20): model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(INFERENCE_RUNS): model(dummy)
        e.record()
        torch.cuda.synchronize()
    return s.elapsed_time(e) / INFERENCE_RUNS

def measure_cpu_ms(model):
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(20): cpu_m(dummy)
    return (time.perf_counter() - t0) / 20 * 1000


from thop import profile

def compute_flops(model, device, input_size=(1, 3, 32, 32)):
    """
    Standard FLOPs computation using THOP.
    Returns:
        flops_total (int): total FLOPs (not MACs)
        flops_G (float): FLOPs in GFLOPs
    """
    model.eval()
    dummy = torch.randn(input_size).to(device)

    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    flops = macs * 2  # convert MACs → FLOPs

    return int(flops), flops / 1e9

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  3. Magnitude Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device: {DEVICE}  |  Comparing: local-L1, local-L2, global-L1")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()
    base_disk = disk_size_mb(model)
    base_flops, base_flops_G = compute_flops(model, DEVICE)

    results = []

    CRITERIA = [
        # ("local_l1",  apply_magnitude_pruning_l1),
        # ("local_l2",  apply_magnitude_pruning_l2),
        ("global_l1", apply_magnitude_pruning_global),
    ]

    best_model = None
    best_score = -1.0
    best_cfg   = None

    for sparsity in SPARSITY_LEVELS:
        for criterion_name, prune_fn in CRITERIA:
            print(f"  Sparsity {int(sparsity*100)}% | Criterion: {criterion_name} ...")
            pruned    = prune_fn(model, sparsity)
            actual_sp = real_sparsity(pruned)
            total_p, active_p = count_params(pruned)
            metrics   = evaluate(pruned, loader)
            inf_gpu   = measure_gpu_ms(pruned)
            inf_cpu   = measure_cpu_ms(pruned)
            acc_drop  = baseline["accuracy"] - metrics["accuracy"]
            sp_mb     = sparse_size_mb(pruned)
            dk_mb     = disk_size_mb(pruned)
            
            pruned_flops, pruned_flops_G = compute_flops(pruned, DEVICE)
    
            row = {
                "target_sparsity"            : sparsity,
                "criterion"                  : criterion_name,
                "actual_sparsity"            : round(actual_sp, 4),
                "accuracy"                   : round(metrics["accuracy"], 6),
                "precision"                  : round(metrics["precision"], 6),
                "recall"                     : round(metrics["recall"], 6),
                "f1"                         : round(metrics["f1"], 6),
                "accuracy_drop"              : round(acc_drop, 6),
                "params_total"               : total_p,
                "params_active"              : active_p,
                "size_sparse_theoretical_mb" : round(sp_mb, 4),
                "size_disk_mb"               : round(dk_mb, 4),
                "disk_saved_mb"              : round(base_disk - dk_mb, 4),
                "inference_gpu_ms"           : round(inf_gpu, 4) if inf_gpu > 0 else None,
                "inference_cpu_ms"           : round(inf_cpu, 4),
                "flops_total"   : int(base_flops),
                "flops_active"  : pruned_flops,
                "flops_reduction_pct": round((1 - pruned_flops / base_flops) * 100, 2)
            }
            results.append(row)

            print(f"    -> Acc: {metrics['accuracy']:.4f} | Drop: {acc_drop:+.4f} | "
                  f"Actual sp: {actual_sp*100:.1f}%")

            if acc_drop <= MAX_ACC_DROP:
                score = metrics["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1
                if score > best_score:
                    best_score = score
                    best_model = copy.deepcopy(pruned)
                    best_cfg   = {"sparsity": sparsity, "criterion": criterion_name}

    report = {
        "method"             : "magnitude_pruning",
        "description"        : "Per-layer magnitude pruning comparing L1-local, L2-local, and global-L1",
        "pruning_granularity": "weight (unstructured)",
        "baseline"           : baseline,
        "pruning_criteria"   : ["local_l1 (per-layer L1)", "local_l2 (per-layer L2^2)", "global_l1 (all layers ranked together)"],
        "device"             : str(DEVICE),
        "max_acc_drop_threshold": MAX_ACC_DROP,
        "best_config"        : best_cfg,
        "results"            : results,
        "notes": {
            "local_l1"  : "Each layer independently pruned to target sparsity — uniform compression",
            "local_l2"  : "Each layer pruned by L2^2 score — prefers keeping large-magnitude weights",
            "global_l1" : "All layers ranked together — important layers auto-protected",
            "comparison_insight": "Global L1 typically outperforms local methods at high sparsity",
        }
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved -> {OUTPUT_JSON}")


if __name__ == "__main__":
    main()



  3. Magnitude Pruning — ResNet-50 / CIFAR-10
  Device: cuda  |  Comparing: local-L1, local-L2, global-L1

  Sparsity 30% | Criterion: global_l1 ...
    -> Acc: 0.9321 | Drop: -0.0001 | Actual sp: 30.0%

  ✓ Saved -> 3_v2_magnitude_Pruning.json
